In [ ]:
#makemore

In [2]:
# download the names.txt file from github
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

--2026-11-03 19:44:41--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.01s   

2026-11-03 19:44:41 (16.7 MB/s) - ‘names.txt’ saved [228145/228145]



In [3]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda")

2.11.0+cu130
True
Tesla T4


In [4]:
file = open('names.txt' , 'r')

In [5]:
names = file.read().split('\n')


In [6]:
chars = sorted(list(set(''.join(names))))
chars.append('.')

stoi = {w:i+1 for i, w in enumerate(chars)}
stoi['.'] = 0
itos = {w:i for i, w in stoi.items()}

# stoi  ,itos

In [7]:
block_size = 3
X = []
Y = []
context = [0] * block_size

for name in names:
  for ch in name:
    ic =  stoi[ch]
    X.append(context)
    Y.append(ic)

    context = context[1:] + [ic]

X = torch.tensor(X , device=device)
Y = torch.tensor(Y , device=device)

# X , Y

In [14]:
batch_size = 5000
epochs = 10_000

g = torch.Generator(device=device).manual_seed(123456789)

C = torch.randn((27,2) , generator=g , device=device)
w1 = torch.randn((6,100) , generator=g , device=device)
b1 = torch.randn((100) , generator=g,  device=device)

w2 = torch.randn((100, 27) , generator=g , device=device)
b2 = torch.randn((27) , generator=g , device=device)

parameters = [C , w1 ,b1 , w2, b2]

for p in parameters:
  p.requires_grad = True

for i in range(epochs):

  ix = torch.randint(low=0, high=X.shape[0] , size=(batch_size,), generator=g, device=device)

  xb = X[ix]
  yb = Y[ix]

  # embeddings
  emb = C[xb]

  # forward pass
  h = torch.tanh(emb.view(-1,6) @ w1 + b1)
  logits = h @ w2 + b2
  loss = torch.nn.functional.cross_entropy(logits , yb)

  # backward pass
  for p in parameters:
    p.grad = None

  loss.backward()

  lr = 0.1 if i < 8000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  if i % 1000 == 0:
    print("Loss : " ,  loss)


Loss :  tensor(15.2913, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss :  tensor(2.7304, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss :  tensor(2.6420, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss :  tensor(2.6530, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss :  tensor(2.6264, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss :  tensor(2.5889, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss :  tensor(2.5825, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss :  tensor(2.6299, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss :  tensor(2.5890, device='cuda:0', grad_fn=<NllLossBackward0>)
Loss :  tensor(2.5913, device='cuda:0', grad_fn=<NllLossBackward0>)
